# GAIL Procurement — Exploratory Data Analysis

Input: cleaned CSVs produced by `01_data_cleaning_feature_engineering.ipynb`

Each section below maps directly to one or more KPIs from the project plan
and to a page in the final 3-page Power BI dashboard:

- **Page 1 (Lifecycle & Bottlenecks):** sections 1-4
- **Page 2 (Vendor Concentration & Risk):** sections 5-6
- **Page 3 (CPBG Compliance & Exposure):** sections 7, 10
- **Supporting analysis:** sections 8-9


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
po = pd.read_csv("po_master_cleaned.csv",
    parse_dates=["PR Date","PO DATE","Schedule Delivery Da","USER HOD Release DT","C&P Release Date"])
cpbg = pd.read_csv("cpbg_info_cleaned.csv",
    parse_dates=["PO Date","Start of Term","Term End","required_term_end"])
diff = pd.read_csv("cpbg_intimation_diff_cleaned.csv")

print(po.shape, cpbg.shape, diff.shape)


## 1. PO Value Distribution

A histogram alone would be misleading here — PO values are heavily
right-skewed (Q3=₹35L but max=₹25.6cr), so we plot on a log scale alongside
a boxplot to show the real spread and outliers together.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vals = po["PO Basic Value (INR)"].dropna()

axes[0].hist(np.log10(vals[vals > 0]), bins=30, color="teal", edgecolor="white")
axes[0].set_title("PO Value Distribution (log10 scale)")
axes[0].set_xlabel("log10(PO Value in INR)")
axes[0].set_ylabel("Count")

sns.boxplot(x=vals, ax=axes[1], color="teal")
axes[1].set_title("PO Value Boxplot (linear scale — outliers visible)")
axes[1].set_xlabel("PO Value (INR)")

plt.tight_layout()
plt.show()

print(f"Median: Rs.{vals.median():,.0f}  |  Mean: Rs.{vals.mean():,.0f}  (skewed — use median for typical PO)")


## 2. PR-to-PO Cycle Time Trend (KPI 1)

Excludes the 6 rows flagged with impossible negative cycle time. No official
GAIL SLA exists for this cycle (confirmed from the C&P Procedure Manual — only
blank tracking templates exist), so this chart *establishes the empirical
baseline* rather than measuring against a target.

In [ ]:
po_valid = po[po["cycle_time_data_flag"] == "OK"].copy()
po_valid["po_month"] = po_valid["PO DATE"].dt.to_period("M").dt.to_timestamp()
trend = po_valid.groupby(["FY", "po_month"])["cycle_days_pr_to_po"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
for fy, grp in trend.groupby("FY"):
    ax.plot(grp["po_month"], grp["cycle_days_pr_to_po"], marker="o", label=fy, linewidth=2)

ax.set_title("PR-to-PO Cycle Time Trend by Month")
ax.set_ylabel("Avg Cycle Days")
ax.set_xlabel("")
ax.legend(title="FY")
plt.tight_layout()
plt.show()

print(po_valid.groupby("FY")["cycle_days_pr_to_po"].agg(["mean", "median", "count"]).round(1))


## 3. Procurement Lifecycle Funnel (KPIs 1, 2)

Simplified to 3 real stages — `USER HOD Release DT` and `C&P Release Date`
were found to be identical in 100% of non-null rows during EDA, so a
4-stage funnel would show a fake "0-day" stage.

In [ ]:
po_valid["pr_to_approval"] = (po_valid["C&P Release Date"] - po_valid["PR Date"]).dt.days
po_valid["approval_to_po"] = (po_valid["PO DATE"] - po_valid["C&P Release Date"]).dt.days
po_valid_delivery = po_valid[po_valid["delivery_data_flag"] == "OK"]
po_to_delivery = po_valid_delivery["po_to_delivery_days"]

stage_means = pd.Series({
    "PR -> Approval\n(HOD+C&P combined)": po_valid["pr_to_approval"].mean(),
    "Approval -> PO": po_valid["approval_to_po"].mean(),
    "PO -> Delivery": po_to_delivery.mean(),
})

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(stage_means.index, stage_means.values, color=sns.color_palette("viridis", 3))
for i, v in enumerate(stage_means.values):
    ax.text(v + 5, i, f"{v:.0f} days", va="center")
ax.set_xlabel("Avg Days")
ax.set_title("Procurement Lifecycle: Avg Days per Stage")
plt.tight_layout()
plt.show()


## 4. Cycle Time by Purchasing Group — Bottleneck Identification

In [ ]:
grp = po_valid.groupby("Purchasing Group")["cycle_days_pr_to_po"].agg(["mean", "count"])
grp = grp[grp["count"] >= 3].sort_values("mean", ascending=False)  # exclude tiny groups

fig, ax = plt.subplots(figsize=(8, 5))
grp["mean"].plot(kind="barh", ax=ax, color="darkcyan")
ax.set_xlabel("Avg Cycle Days")
ax.set_title("Cycle Time by Purchasing Group (groups with 3+ POs)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Vendor Concentration (KPI 9)

HHI (Herfindahl-Hirschman Index) trend, plus the top-15 vendors by spend.

In [ ]:
vendor_totals = po.groupby(["FY", "VENDOR CODE"])["PO Basic Value (INR)"].sum().reset_index()
vendor_totals["share"] = vendor_totals.groupby("FY")["PO Basic Value (INR)"].transform(lambda x: x / x.sum())
hhi = vendor_totals.groupby("FY").apply(lambda d: (d["share"]**2).sum() * 10000, include_groups=False).reset_index(name="HHI")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hhi["FY"], hhi["HHI"], marker="o", color="crimson", linewidth=2, markersize=8)
ax.axhline(2500, color="gray", linestyle="--", alpha=0.7, label="High concentration threshold (2500)")
ax.set_title("Vendor Concentration Index (HHI) by FY")
ax.set_ylabel("HHI")
ax.legend()
plt.tight_layout()
plt.show()
print(hhi)


In [ ]:
top_vendors = po.groupby("VENDOR CODE")["PO Basic Value (INR)"].sum().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
top_vendors.iloc[::-1].plot(kind="barh", ax=ax, color=sns.color_palette("viridis", 15))
ax.set_xlabel("Total PO Value (INR)")
ax.set_title("Top 15 Vendors by Total Spend")
plt.tight_layout()
plt.show()


## 6. Vendor Risk: Delay Rate vs Deviation (KPI 10 inputs)

Bubble chart — vendors with 2+ POs only (single-PO vendors have too little
data to trust a "risk" label). Bubble size = total spend, so you can see at a
glance which risky vendors also carry high spend exposure.

In [ ]:
po["exceeded_std"] = po["lead_time_deviation_days"] > 0
vendor_risk = po.groupby("VENDOR CODE").agg(
    n_pos=("PO NO", "count"),
    delay_rate=("exceeded_std", "mean"),
    avg_deviation=("lead_time_deviation_days", "mean"),
    total_spend=("PO Basic Value (INR)", "sum"),
).reset_index()
vendor_risk = vendor_risk[vendor_risk["n_pos"] >= 2]

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    vendor_risk["delay_rate"] * 100, vendor_risk["avg_deviation"],
    s=vendor_risk["total_spend"] / vendor_risk["total_spend"].max() * 800 + 20,
    c=vendor_risk["total_spend"], cmap="viridis", alpha=0.7, edgecolor="k", linewidth=0.3,
)
ax.set_xlabel("Delay Rate (%)")
ax.set_ylabel("Avg Lead Time Deviation (days)")
ax.set_title("Vendor Risk: Delay Rate vs Deviation (bubble size = spend)")
plt.colorbar(sc, label="Total Spend (INR)")
plt.tight_layout()
plt.show()

vendor_risk.sort_values("delay_rate", ascending=False).head(10)


## 7. CPBG Compliance Rates (KPIs 4, 5, 6, 7)

The headline finding of the whole project: value and duration compliance are
reasonably high, but intimation compliance is very low, every single year —
and this is measured against an explicit, cited clause in GAIL's own
procedure (3.2.1 Note iii), not an assumption.

In [ ]:
value_comp = cpbg.groupby("FY")["cpbg_value_compliant"].mean() * 100
dur_comp = cpbg.groupby("FY")["cpbg_duration_compliant"].mean() * 100
diff["intimation_compliant"] = diff["Difference_num"] <= 30
intim_comp = diff.groupby("FY")["intimation_compliant"].mean() * 100

comp_df = pd.DataFrame({
    "Value Compliance": value_comp,
    "Duration Compliance": dur_comp,
    "Intimation Compliance": intim_comp,
}).fillna(0)

fig, ax = plt.subplots(figsize=(9, 5))
comp_df.plot(kind="bar", ax=ax, color=["#2a9d8f", "#e9c46a", "#e76f51"])
ax.axhline(100, color="gray", linestyle=":", alpha=0.7)
ax.set_ylabel("Compliance Rate (%)")
ax.set_title("CPBG Compliance Rates by FY (per C&P Procedure Clauses)")
ax.set_xticklabels(comp_df.index, rotation=0)
plt.tight_layout()
plt.show()

comp_df.round(1)


## 8. MSME Category Spend Share (KPI 11)

In [ ]:
msme = po.groupby(["FY", "msme_category"])["PO Basic Value (INR)"].sum().unstack(fill_value=0)
msme_pct = msme.div(msme.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
msme_pct.plot(kind="bar", stacked=True, ax=ax, color=sns.color_palette("viridis", 4))
ax.set_ylabel("% of Total Spend")
ax.set_title("MSME Category Spend Share by FY")
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print("Combined Micro+Small+Medium share:")
print((msme_pct[["Micro", "Small", "Medium"]].sum(axis=1)).round(1))


## 9. Correlation Check — Cycle Time, Lag, Deviation, Value

In [ ]:
numeric_cols = ["cycle_days_pr_to_po", "approval_lag_days", "lead_time_deviation_days", "PO Basic Value (INR)"]
corr = po[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
ax.set_title("Correlation: Cycle Time, Lag, Deviation, Value")
plt.tight_layout()
plt.show()


## 10. BG Exposure Timeline (KPI 8 — "Expiry Radar")

Each horizontal line is one guarantee's active term. The red dashed line is
today — guarantees whose bar crosses it are currently live exposure; bars
ending soon after today are the ones to watch for upcoming renewal/release.

In [ ]:
cpbg_valid = cpbg.dropna(subset=["Start of Term", "Term End"]).sort_values("Start of Term").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 8))
for i, row in cpbg_valid.iterrows():
    ax.plot([row["Start of Term"], row["Term End"]], [i, i], color="steelblue", linewidth=3, alpha=0.7)

today = pd.Timestamp.now()
ax.axvline(today, color="red", linestyle="--", label="Today")
ax.set_yticks([])
ax.set_ylabel(f"{len(cpbg_valid)} guarantees (each row = one CPBG)")
ax.set_title("CPBG Term Timeline (Start of Term -> Term End)")
ax.legend()
plt.tight_layout()
plt.show()

live_now = cpbg_valid[(cpbg_valid["Start of Term"] <= today) & (cpbg_valid["Term End"] >= today)]
print(f"Currently live guarantees: {len(live_now)}, total exposure: Rs.{live_now['Credit Amount'].sum():,.0f}")


## Summary of Key Findings

- **PR-to-PO cycle time worsened** after 2023-24, with no official SLA to
  benchmark against (confirmed absent from the C&P Procedure Manual itself).
- **The HOD Release and C&P Release dates are identical in every record** —
  worth raising as a data-capture observation, not just a chart footnote.
- **Vendor concentration (HHI) swings sharply** year to year (1243 -> 235 -> 1114).
- **CPBG value and duration compliance are reasonably strong (~73-86%)**, but
  **intimation compliance is consistently low (12-21%)** across all three
  years — the strongest, most defensible finding in the whole project, since
  it's measured directly against a cited clause in GAIL's own procedure.
- **MSME combined spend share is well above typical mandates**, a genuinely
  positive finding worth highlighting.

Next: use these findings and the KPI list to build the 3-page Power BI
dashboard following the page-by-page plan already documented.